In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI

openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)    

In [4]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Usually, yes—**if the course is still open for enrollment**.\n\nA few things to check:\n- **Enrollment deadline**: some courses close registration after the start date.\n- **Prerequisites**: you may need prior knowledge or approval.\n- **Capacity**: the class may be full.\n- **Platform rules**: some courses allow late enrollment, others don’t.\n\nIf you want, I can help you draft a short message to the course instructor or registrar asking whether you can still join.'

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course discovered late enrollment can I join it"}', call_id='call_pjmwo04ZUEPdFMF4c1rq1kR8', name='search', type='function_call', id='fc_04c3cd6763be3d20006a32d8a891c081a38519b37afeba93cb', namespace=None, status='completed')]

In [8]:
call = response.output[0]

In [9]:
call

ResponseFunctionToolCall(arguments='{"query":"join the course discovered late enrollment can I join it"}', call_id='call_pjmwo04ZUEPdFMF4c1rq1kR8', name='search', type='function_call', id='fc_04c3cd6763be3d20006a32d8a891c081a38519b37afeba93cb', namespace=None, status='completed')

In [10]:
call.type

'function_call'

In [11]:
call.call_id

'call_pjmwo04ZUEPdFMF4c1rq1kR8'

In [12]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [13]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course

In [14]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [15]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course discovered late enrollment can I join it"}', call_id='call_pjmwo04ZUEPdFMF4c1rq1kR8', name='search', type='function_call', id='fc_04c3cd6763be3d20006a32d8a891c081a38519b37afeba93cb', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_pjmwo04ZUEPdFMF4c1rq1kR8',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\

In [16]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [17]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open.


In [18]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [20]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late join registration"}


In [21]:
has_function_calls

True

In [22]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes, you can still join the course. If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.

If you’re okay with learning without the certificate, you can still follow the materials and start anytime.

Would you like to know more about certificates, deadlines, or how to start the course?


In [23]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [24]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install local model Ollama run locally"}
function_call: search {"query":"Ollama local setup run model locally install start server"}
function_call: search {"query":"run Ollama locally FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - **macOS / Windows:** download it from https://ollama.com/download
   - **Linux:**
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s working, you should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - **macOS / Windows:** download it from https://ollama.com/download\n   - **Linux:**\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Then:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also show you how to run Ollama with

In [25]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
iteration #2...
ASSISTANT:
Yes, you can still join the course.

If you want to receive a certificate, though, you need to submit your project while submissions are still being accepted.

If you’d like, I can also help you understand whether you can still get a certificate, how homework works, or what to do next after joining.


'Yes, you can still join the course.\n\nIf you want to receive a certificate, though, you need to submit your project while submissions are still being accepted.\n\nIf you’d like, I can also help you understand whether you can still get a certificate, how homework works, or what to do next after joining.'

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered just discovered can I join course enrollment late join"}
iteration #2...
function_call: search {"query":"certificate join course while accepting submissions live cohort self-paced late submission peer-review accepted enroll"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, make sure you submit your project while submissions are still open.

Also, if you want, I can help you figure out how to start from where you are now or explain the certificate requirements. Any other area you’d like to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, though, make sure you submit your project while submissions are still open.\n\nAlso, if you want, I can help you figure out how to start from where you are now or explain the certificate requirements. Any other area you’d like to explore?'

In [27]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to gain control of the center. It’s called a “gambit” because a pawn is offered, though in practice Black often doesn’t keep it for long.

A common follow-up is:
- **1. d4 d5 2. c4**

If you meant the **Netflix show "The Queen’s Gambit"**, I can explain that too. Want the chess version in more detail?


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to gain control of the center. It’s called a “gambit” because a pawn is offered, though in practice Black often doesn’t keep it for long.\n\nA common follow-up is:\n- **1. d4 d5 2. c4**\n\nIf you meant the **Netflix show "The Queen’s Gambit"**, I can explain that too. Want the chess version in more detail?'

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening queen gambit FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this is off-topic for the course database.

If you meant a course-related term or logistics question, feel free to rephrase it. Is there another area you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this is off-topic for the course database.\n\nIf you meant a course-related term or logistics question, feel free to rephrase it. Is there another area you want to explore?'